In [1]:
!pip install -q ultralytics opencv-python numpy torch
!pip install -q onnx onnxslim onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.7/238.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 71.6 MB/s eta 0:00:00


In [2]:
import os, time, copy
os.environ["PYTHONUNBUFFERED"] = "1"
import numpy as np
import cv2
import torch
import yaml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ultralytics import YOLO

OUT = "/kaggle/working/mlops_report"
os.makedirs(OUT, exist_ok=True)

# ============ 0. FILE CẤU HÌNH NGƯỠNG (THEO SPEC) ============
CONFIG = {
    "map_small_min": 0.40,
    "critical_fnr_max": 0.02,
    "dbbr_max": 0.03,
    "multilabel_miss_max": 0.05,
    "illumination_drop_max": 0.10,
    "forgetting_delta_max": 0.03,
    "p99_latency_max_ms": 100.0,
    "psi_alert": 0.2,
    "ucr_max": 0.05,
}
CFG_PATH = f"{OUT}/metrics_config.yaml"
with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, sort_keys=False, allow_unicode=True)
with open(CFG_PATH, encoding="utf-8") as f:
    TH = yaml.safe_load(f)
print(f"✅ File cấu hình ngưỡng: {CFG_PATH}", flush=True)

DUAL_DEVICE, SINGLE_DEVICE = "0,1", "0"
BATCH_DUAL, BATCH_SINGLE = 86, 32
CRITICAL_CLASS_NAMES = []
BASELINE_MAP_OLD = 0.5252

# ============ 1. QUÉT INPUT NHANH (PRUNE images/labels) ============
def scan_inputs():
    PT, test_cand, yaml_cand, img_dir = {}, [], [], None
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in ("images", "labels")]
        lr = root.lower()
        if lr.endswith("unified_dataset"):
            ci, cl = os.path.join(root, "images"), os.path.join(root, "labels")
            if os.path.isdir(ci) and os.path.isdir(cl) and img_dir is None:
                img_dir = ci
        for fn in files:
            fp = os.path.join(root, fn)
            if fn == "best.pt":
                for acc in (1, 2, 3):
                    if f"acc{acc}" in lr:
                        PT[acc] = fp
            if fn == "fixed_test.txt":
                test_cand.append((0, fp))
            elif fn == "test.txt":
                test_cand.append((1, fp))
            if fn == "fixed_data_account_1.yaml":
                yaml_cand.append((0, fp))
            elif fn == "data.yaml":
                yaml_cand.append((1, fp))
    return PT, (min(test_cand)[1] if test_cand else None), (min(yaml_cand)[1] if yaml_cand else None), img_dir

PT, TEST_TXT, DATA_YAML, IMG_DIR = scan_inputs()
assert len(PT) == 3 and TEST_TXT and DATA_YAML and IMG_DIR, f"Thiếu Input! PT={PT}, IMG={IMG_DIR}"
print(f"✅ ACC1: {PT[1]}\n✅ ACC2: {PT[2]}\n✅ ACC3: {PT[3]}", flush=True)
print(f"✅ IMG_DIR: {IMG_DIR}", flush=True)

with open(TEST_TXT, encoding="utf-8") as f:
    raw = [l.strip() for l in f if l.strip()]
if raw and not os.path.exists(raw[0]) and os.path.exists(os.path.join(IMG_DIR, os.path.basename(raw[0]))):
    raw = [os.path.join(IMG_DIR, os.path.basename(p)) for p in raw]
    TEST_TXT = f"{OUT}/test_fixed_paths.txt"
    with open(TEST_TXT, "w") as f:
        f.write("\n".join(raw))
    print("✅ Đã chuẩn hóa đường dẫn ảnh", flush=True)

for p in raw[:5]:
    lp = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    assert os.path.exists(p) and os.path.exists(lp), f"Thiếu ảnh/label: {p}"
print("✅ Ảnh + label sẵn sàng", flush=True)
test_paths = raw

# ============ 2. YAML HỢP LỆ ============
with open(DATA_YAML, encoding="utf-8") as f:
    ydata = yaml.safe_load(f)
ydata["test"] = TEST_TXT
ydata.pop("path", None)
FIXED_YAML = f"{OUT}/fixed_eval.yaml"
with open(FIXED_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(ydata, f, sort_keys=False, allow_unicode=True)
_names = ydata["names"]
if isinstance(_names, dict):
    _names = [_names[k] for k in sorted(_names)]
name2idx = {n.lower(): i for i, n in enumerate(_names)}
critical_ids = set(name2idx[c.lower()] for c in CRITICAL_CLASS_NAMES if c.lower() in name2idx)

# ============ 3. MODEL SOUP (CPU) ============
print("⏳ Hợp nhất 3 model...", flush=True)
sds, base_obj = [], None
for acc in (1, 2, 3):
    ckpt = torch.load(PT[acc], map_location="cpu", weights_only=False)
    obj = ckpt["ema"] if isinstance(ckpt, dict) and ckpt.get("ema") is not None else (ckpt["model"] if isinstance(ckpt, dict) else ckpt)
    obj = obj.float()
    if base_obj is None:
        base_obj = obj
    sds.append(obj.state_dict())
merged_sd = copy.deepcopy(sds[0])
for key in merged_sd:
    if merged_sd[key].dtype.is_floating_point:
        merged_sd[key] = torch.stack([s[key] for s in sds]).mean(dim=0)
merged_model = copy.deepcopy(base_obj)
merged_model.load_state_dict(merged_sd, strict=True)
merged_model.eval()
torch.save({"model": merged_model}, "/kaggle/working/best_merged.pt")
model = YOLO("/kaggle/working/best_merged.pt")
print("✅ best_merged.pt sẵn sàng", flush=True)

# ============ 4. SAFE VAL (DUAL T4 + FALLBACK OOM) ============
def safe_val(data):
    try:
        return model.val(data=data, split="test", device=DUAL_DEVICE, batch=BATCH_DUAL, plots=False, verbose=False)
    except Exception as e:
        print(f"⚠️ Dual-GPU lỗi ({str(e)[:60]}), fallback single...", flush=True)
        torch.cuda.empty_cache()
        return model.val(data=data, split="test", device=SINGLE_DEVICE, batch=BATCH_SINGLE, plots=False, verbose=False)

print("⏳ [Main] Val toàn bộ test set...", flush=True)
val_results = safe_val(FIXED_YAML)
md = val_results.results_dict
overall_map = md["metrics/mAP50-95(B)"]
map50 = md["metrics/mAP50(B)"]
prec = md["metrics/precision(B)"]
rec = md["metrics/recall(B)"]

# ============ 5. PSI + ILLUMINATION SLICES ============
print("⏳ [A] Quét độ sáng...", flush=True)
FLAG_READ = getattr(cv2, "IMREAD_REDUCED_RESOLUTION_2", 1)  # thiếu hằng số -> đọc màu thường
bright = []
for p in test_paths:
    img = cv2.imread(p, FLAG_READ)
    bright.append(-1.0 if img is None else float(cv2.cvtColor(cv2.resize(img, (64, 64)), cv2.COLOR_BGR2HSV)[:, :, 2].mean()))
vals_ok = [b for b in bright if b >= 0]
h_val, _ = np.histogram(vals_ok, bins=10, range=(0, 255))
h_val = np.clip(h_val.astype(np.float64) / max(sum(h_val), 1), 1e-4, None)
h_train = np.clip(np.ones(10) / 10, 1e-4, None)
psi = float(np.sum((h_val - h_train) * np.log(h_val / h_train)))

thr = np.percentile(sorted(vals_ok), [33.3, 66.6])
slices = {"Thieu_sang": [], "Du_sang": [], "Chay_sang": []}
for p, b in zip(test_paths, bright):
    if b >= 0:
        slices["Thieu_sang" if b < thr[0] else ("Du_sang" if b < thr[1] else "Chay_sang")].append(p)
slice_maps = {}
for sname, plist in slices.items():
    if not plist:
        continue
    stxt = f"{OUT}/test_slice_{sname}.txt"
    with open(stxt, "w") as f:
        f.write("\n".join(plist))
    ys = yaml.safe_load(open(FIXED_YAML, encoding="utf-8"))
    ys["test"] = stxt
    syp = f"{OUT}/yaml_slice_{sname}.yaml"
    with open(syp, "w", encoding="utf-8") as f:
        yaml.safe_dump(ys, f, sort_keys=False)
    slice_maps[sname] = safe_val(syp).results_dict["metrics/mAP50-95(B)"]
drops = [(overall_map - v) / max(overall_map, 1e-9) * 100 for v in slice_maps.values()]

# ============ 6. METRIC MỨC HỘP (800 MẪU, BATCH 16) ============
print("⏳ [B] Metric mức hộp...", flush=True)
def iou(a, b):
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

rng = np.random.default_rng(42)
sample = [test_paths[i] for i in rng.choice(len(test_paths), size=min(800, len(test_paths)), replace=False)]
tp = fp = fn = 0
per_gt = np.zeros(len(_names), dtype=np.int64)
per_tp = np.zeros(len(_names), dtype=np.int64)
small_gt = small_tp = crit_gt = crit_tp = 0
merge_gt = dup_pred = tot_gt = tot_pred = 0
multi_miss = multi_tot = ood_cnt = 0
mae_all = []
strata = {"Thua_<5": [], "Trung_5_20": [], "Day_>20": []}

for i in range(0, len(sample), 16):
    for res in model.predict(sample[i:i+16], imgsz=640, conf=0.001, verbose=False, device=SINGLE_DEVICE):
        p = res.path
        H, W = res.orig_shape
        lp = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        gts = []
        if os.path.exists(lp):
            for line in open(lp):
                a = line.split()
                if len(a) >= 5:
                    c = int(float(a[0])); x, y, w, h = map(float, a[1:5])
                    gts.append([c, (x-w/2)*W, (y-h/2)*H, (x+w/2)*W, (y+h/2)*H])
        preds = []
        if res.boxes is not None and len(res.boxes):
            for b in res.boxes:
                x1, y1, x2, y2 = b.xyxy[0].tolist()
                preds.append([int(b.cls[0]), x1, y1, x2, y2, float(b.conf[0])])

        tot_gt += len(gts); tot_pred += len(preds)
        pairs = sorted([(iou(pr[1:5], g[1:5]), pi, gi) for pi, pr in enumerate(preds) for gi, g in enumerate(gts) if pr[0] == g[0] and iou(pr[1:5], g[1:5]) >= 0.5], key=lambda t: -t[0])
        mp_, mg_ = {}, {}
        for v, pi, gi in pairs:
            if pi not in mp_ and gi not in mg_:
                mp_[pi] = gi; mg_[gi] = pi
        tp += len(mg_); fp += len(preds) - len(mp_); fn += len(gts) - len(mg_)
        for gi, g in enumerate(gts):
            per_gt[g[0]] += 1
            if (g[3]-g[1]) * (g[4]-g[2]) < 1024:
                small_gt += 1
            if g[0] in critical_ids:
                crit_gt += 1
                if gi in mg_:
                    crit_tp += 1
        for gi in mg_:
            per_tp[gts[gi][0]] += 1
            if (gts[gi][3]-gts[gi][1]) * (gts[gi][4]-gts[gi][2]) < 1024:
                small_tp += 1
        err = abs(len(preds) - len(gts))
        mae_all.append(err)
        n = len(gts)
        strata["Thua_<5" if n < 5 else ("Trung_5_20" if n <= 20 else "Day_>20")].append(err)
        for pr in preds:
            if sum(1 for g in gts if g[0] == pr[0] and iou(pr[1:5], g[1:5]) >= 0.3) >= 2:
                merge_gt += 1
        for pi, pr in enumerate(preds):
            if pi not in mp_ and any(g[0] == pr[0] and iou(pr[1:5], g[1:5]) >= 0.5 for g in gts):
                dup_pred += 1
        cls_present = sorted(set(g[0] for g in gts))
        if len(cls_present) >= 2:
            detected = set(gts[gi][0] for gi in mg_.values())
            for c in cls_present[1:]:
                multi_tot += 1
                if c not in detected:
                    multi_miss += 1
        if not any(pr[5] >= 0.25 for pr in preds):
            ood_cnt += 1

P2 = tp / max(tp + fp, 1)
R2 = tp / max(tp + fn, 1)
F2 = 5 * P2 * R2 / max(4 * P2 + R2, 1e-9)
recalls = np.where(per_gt > 0, per_tp / np.maximum(per_gt, 1), np.nan)
worst_recall = float(np.nanmin(recalls)) if (~np.isnan(recalls)).any() else 0.0
gmean = float(np.exp(np.nanmean(np.log(recalls + 1e-8)))) if (~np.isnan(recalls)).any() else 0.0
map_small = small_tp / max(small_gt, 1)
dbbr = dup_pred / max(tot_pred, 1)
mlm = multi_miss / max(multi_tot, 1) if multi_tot else 0.0
fnr = 1 - crit_tp / max(crit_gt, 1) if crit_gt else None
forget = BASELINE_MAP_OLD - overall_map

# ============ 7. BENCHMARK + RAM ============
print("⏳ [C] Benchmark...", flush=True)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
dev = torch.device("cuda:0")
model.model.to(dev).half().eval()
x = torch.randn(1, 3, 640, 640, device=dev, dtype=torch.float16)
with torch.no_grad():
    for _ in range(20):
        _ = model.model(x)
torch.cuda.synchronize()
lats = []
with torch.no_grad():
    for _ in range(50):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model.model(x)
        torch.cuda.synchronize()
        lats.append((time.perf_counter() - t0) * 1000)
p50, p95, p99 = np.percentile(lats, [50, 95, 99])
peak_ram = torch.cuda.max_memory_allocated() / 1e6

# ============ 8. EXPORT ONNX ============
print("⏳ [D] Export ONNX...", flush=True)
m2 = YOLO("/kaggle/working/best_merged.pt")
try:
    onnx_path = m2.export(format="onnx", half=True, simplify=True, opset=12, device=SINGLE_DEVICE)
except Exception:
    onnx_path = m2.export(format="onnx", half=False, simplify=True, opset=12, device=SINGLE_DEVICE)

# ============ 9. VISUALIZE ALL METRICS ============
print("⏳ [E] Vẽ biểu đồ...", flush=True)
def save(name):
    plt.tight_layout()
    plt.savefig(f"{OUT}/{name}.png", dpi=150)
    plt.close()

plt.figure(figsize=(10, 5))
plt.bar(["mAP50-95", "mAP50", "Precision", "Recall", "F2", "mAP_small", "WorstRec", "G-Mean"],
        [overall_map, map50, prec, rec, F2, map_small, worst_recall, gmean], color="#2e7d32")
plt.ylim(0, 1)
plt.axhline(TH["map_small_min"], color="red", ls="--", label="ngưỡng mAP_small")
plt.legend(); plt.xticks(rotation=20); plt.title("Nhóm 1: Chất lượng mô hình")
save("1_model_quality")

plt.figure(figsize=(8, 5))
plt.bar(["MAE", "ClusterMerge", "DBBR", "MultiMiss"],
        [float(np.mean(mae_all)), merge_gt / max(tot_gt, 1), dbbr, mlm], color="#1565c0")
plt.plot([1.6, 2.4], [TH["dbbr_max"]]*2, "r--")
plt.plot([2.6, 3.4], [TH["multilabel_miss_max"]]*2, "r--")
plt.title("Nhóm 2: Đếm mật độ & lỗi hộp"); plt.xticks(rotation=15)
save("2_bug_count")

plt.figure(figsize=(8, 5))
ns = list(slice_maps.keys())
vs = [slice_maps[k] for k in ns]
plt.bar(ns, vs, color="#ef6c00")
plt.axhline(overall_map, color="green", label=f"Overall {overall_map:.4f}")
for i, d in enumerate(drops):
    plt.text(i, vs[i] + 0.005, f"-{d:.1f}%", ha="center", color="red")
plt.legend(); plt.title("Nhóm 3: Illumination-Sliced mAP (sụt <10%)")
save("3_illumination")

plt.figure(figsize=(8, 5))
plt.hist(vals_ok, bins=30, range=(0, 255), color="#6a1b9a", alpha=0.8)
plt.title(f"Nhóm 3: PSI = {psi:.4f} ({'ALERT' if psi > TH['psi_alert'] else 'STABLE'})")
plt.xlabel("Brightness (V)")
save("4_psi_histogram")

plt.figure(figsize=(8, 5))
plt.bar(["p50", "p95", "p99"], [p50, p95, p99], color="#00838f")
plt.axhline(TH["p99_latency_max_ms"], color="red", ls="--", label="ngưỡng 100ms")
plt.legend(); plt.ylabel("ms"); plt.title("Nhóm 4: Edge Latency FP16")
save("5_latency")

plt.figure(figsize=(10, 5))
plt.plot(np.sort([per_tp[i] / per_gt[i] for i in np.where(per_gt > 0)[0]]), color="#2e7d32")
plt.axhline(worst_recall, color="red", ls="--", label=f"Worst={worst_recall:.3f}")
plt.legend(); plt.title("Nhóm 1: Per-Class Recall (sorted)")
save("6_per_class_recall")

ks = [k for k, v in strata.items() if v]
plt.figure(figsize=(7, 5))
plt.bar(ks, [np.mean(strata[k]) for k in ks], color="#455a64")
plt.title("Nhóm 2: Density-Stratified MAE")
save("7_mae_strata")

def st(c):
    return "✅" if c else "❌"
rows = [
    ["mAP@0.5:0.95", f"{overall_map:.4f}", "-", "✅"],
    ["mAP_small", f"{map_small:.4f}", f"> {TH['map_small_min']}", st(map_small > TH["map_small_min"])],
    ["Critical FNR", f"{fnr:.4f}" if fnr is not None else "N/A", f"< {TH['critical_fnr_max']}", st(fnr < TH["critical_fnr_max"]) if fnr is not None else "-"],
    ["F2-Score", f"{F2:.4f}", "-", "✅"],
    ["Worst-Class Recall", f"{worst_recall:.4f}", "-", "-"],
    ["DBBR", f"{dbbr:.4f}", f"< {TH['dbbr_max']}", st(dbbr < TH["dbbr_max"])],
    ["Multi-label Miss", f"{mlm:.4f}", f"< {TH['multilabel_miss_max']}", st(mlm < TH["multilabel_miss_max"])],
    ["Illumination drop", f"{max(drops) if drops else 0:.1f}%", f"< {TH['illumination_drop_max']*100:.0f}%", st((max(drops) if drops else 0) / 100 < TH["illumination_drop_max"])],
    ["PSI Drift", f"{psi:.4f}", f"< {TH['psi_alert']}", st(psi < TH["psi_alert"])],
    ["Forgetting Delta", f"{forget:+.4f}", f"< {TH['forgetting_delta_max']}", st(abs(forget) < TH["forgetting_delta_max"])],
    ["p99 Latency", f"{p99:.2f} ms", f"< {TH['p99_latency_max_ms']:.0f} ms", st(p99 < TH["p99_latency_max_ms"])],
    ["UCR", "N/A", f"< {TH['ucr_max']}", "-"],
]
fig, ax = plt.subplots(figsize=(10, 7))
ax.axis("off")
ax.table(cellText=rows, colLabels=["Metric", "Giá trị", "Ngưỡng", "Đạt"], loc="center", cellLoc="center")
ax.set_title("BÁO CÁO MLOPS - MODEL HỢP NHẤT (3 ACC)")
plt.savefig(f"{OUT}/0_summary_table.png", dpi=150, bbox_inches="tight")
plt.close()

# ============ 10. BÁO CÁO CUỐI ============
print("\n" + "=" * 64, flush=True)
print("📊 HOÀN TẤT - FILE BÁO CÁO:", flush=True)
for fn in sorted(os.listdir(OUT)):
    print(f"   - {OUT}/{fn}", flush=True)
print(f"📦 ONNX duy nhất: {onnx_path}", flush=True)
print("=" * 64, flush=True)
print("🎉 PIPELINE HOÀN TẤT.", flush=True)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ File cấu hình ngưỡng: /kaggle/working/mlops_report/metrics_config.yaml
✅ ACC1: /kaggle/input/datasets/vdt1501/acc1-best-onnx-mlops-sau-benh-cay-trong/best.pt
✅ ACC2: /kaggle/input/datasets/vdt1501/acc2-best-onnx-mlops-sau-benh-cay-trong/best.pt
✅ ACC3: /kaggle/input/datasets/vdt1501/acc3-best-onnx-mlops-sau-benh-cay-trong/best.pt
✅ IMG_DIR: /kaggle/input/datasets/vdt1501/unified-disease-leaf-ip102/unified_dataset/images
✅ Ảnh + label sẵn sàng
⏳ Hợp nhất 3 model...
✅ best_merged.pt sẵn sàng
⏳ [Main] Val toàn bộ test set...
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 1

/tmp/ipykernel_23/4027292924.py:366: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(f"{OUT}/0_summary_table.png", dpi=150, bbox_inches="tight")
/tmp/ipykernel_23/4027292924.py:366: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(f"{OUT}/0_summary_table.png", dpi=150, bbox_inches="tight")



📊 HOÀN TẤT - FILE BÁO CÁO:
   - /kaggle/working/mlops_report/0_summary_table.png
   - /kaggle/working/mlops_report/1_model_quality.png
   - /kaggle/working/mlops_report/2_bug_count.png
   - /kaggle/working/mlops_report/3_illumination.png
   - /kaggle/working/mlops_report/4_psi_histogram.png
   - /kaggle/working/mlops_report/5_latency.png
   - /kaggle/working/mlops_report/6_per_class_recall.png
   - /kaggle/working/mlops_report/7_mae_strata.png
   - /kaggle/working/mlops_report/fixed_eval.yaml
   - /kaggle/working/mlops_report/metrics_config.yaml
   - /kaggle/working/mlops_report/test_slice_Chay_sang.txt
   - /kaggle/working/mlops_report/test_slice_Du_sang.txt
   - /kaggle/working/mlops_report/test_slice_Thieu_sang.txt
   - /kaggle/working/mlops_report/yaml_slice_Chay_sang.yaml
   - /kaggle/working/mlops_report/yaml_slice_Du_sang.yaml
   - /kaggle/working/mlops_report/yaml_slice_Thieu_sang.yaml
📦 ONNX duy nhất: /kaggle/working/best_merged.onnx
🎉 PIPELINE HOÀN TẤT.
